In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, Embedding
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import pandas as pd
import numpy as np
import random

print("Packages loaded successfully")


c:\Users\ellio\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Packages loaded successfully


In [2]:
# Load data (last 210 columns are binary targets)
features_df = pd.read_csv(r"C:\Users\ellio\git\gcn-dmpc\final-analysis\first-6-decompositions\MLE-classifier-on-1-6\ML-features.csv")

num_nodes = 21
num_targets = 210

# Architecture / training knobs
SELF_LOOP_WEIGHT = 5.0
MODEL_VARIANT = "gat1"   # "gat1", "gat2", or "nogat"
USE_NODE_EMB = True
NODE_EMB_DIM = 8
CONCAT_RAW = True
HIDDEN_CHANNELS = 256
HEADS = 4
MLP_HIDDEN = [512, 256]
DROPOUT = 0.1

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if hasattr(torch, "use_deterministic_algorithms"):
    torch.use_deterministic_algorithms(True, warn_only=True)
print(f"Using SEED={SEED}")


# Preserve original column order: first 21 columns are node features, last 210 columns are targets
feature_cols = features_df.columns[:num_nodes]
target_cols = features_df.columns[-num_targets:]

if len(feature_cols) != num_nodes:
    raise ValueError(f"Expected {num_nodes} feature columns, got {len(feature_cols)}")
if len(target_cols) != num_targets:
    raise ValueError(f"Expected {num_targets} target columns, got {len(target_cols)}")

# 80/20 split without shuffling (by row order)
split_idx = int(0.8 * len(features_df))
train_df = features_df.iloc[:split_idx].copy()
test_df = features_df.iloc[split_idx:].copy()

train_features = train_df[feature_cols].values
train_targets = train_df[target_cols].values

test_features = test_df[feature_cols].values
test_targets = test_df[target_cols].values

# Keep edge_list as provided (1-based node ids, weight is third entry)
edge_list = [(1, 4, 1), (1, 7, 1), (1, 10, 1), (1, 13, 1), (1, 16, 1), (1, 18, 1), (2, 5, 1), (2, 8, 1), (2, 11, 1), (2, 13, 1), (2, 14, 1), (2, 17, 1), (3, 6, 1), (3, 9, 1), (3, 12, 1), (3, 14, 1), (3, 15, 1), (3, 18, 1), (4, 5, 1), (4, 6, 1), (4, 7, 1), (4, 10, 1), (4, 16, 1), (4, 18, 1), (4, 19, 1), (5, 6, 1), (5, 8, 1), (5, 11, 1), (5, 13, 1), (5, 17, 1), (5, 20, 1), (6, 14, 1), (6, 21, 1), (7, 8, 1), (7, 9, 1), (7, 10, 1), (7, 12, 1), (7, 16, 1), (7, 18, 1), (8, 9, 1), (8, 11, 1), (8, 13, 1), (8, 17, 1), (9, 10, 1), (9, 12, 1), (9, 14, 1), (9, 18, 1), (10, 11, 1), (10, 12, 1), (10, 16, 1), (10, 18, 1), (11, 12, 1), (11, 13, 1), (11, 17, 1), (12, 14, 1), (12, 18, 1)]


Using SEED=42


In [3]:
# Build undirected edge_index and edge_attr (use third entry as edge weight)
edge_index_pairs = []
edge_attr_values = []
for u, v, w in edge_list:
    u_idx, v_idx = u - 1, v - 1
    edge_index_pairs.append([u_idx, v_idx])
    edge_attr_values.append([w])
    edge_index_pairs.append([v_idx, u_idx])
    edge_attr_values.append([w])

# Add self-loops with configurable weight
for n in range(num_nodes):
    edge_index_pairs.append([n, n])
    edge_attr_values.append([SELF_LOOP_WEIGHT])

edge_index = torch.tensor(edge_index_pairs, dtype=torch.long).t().contiguous()
edge_attr = torch.tensor(edge_attr_values, dtype=torch.float32)

print(f"edge_index shape: {edge_index.shape}")
print(f"edge_attr shape: {edge_attr.shape}")

# Build graph datasets (each row is a graph with 21 nodes and 210 binary targets)
train_graphs = []
for row, y in zip(train_features, train_targets):
    x = torch.tensor(row, dtype=torch.float32).view(num_nodes, 1)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
    train_graphs.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor))

test_graphs = []
for row, y in zip(test_features, test_targets):
    x = torch.tensor(row, dtype=torch.float32).view(num_nodes, 1)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
    test_graphs.append(Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor))

train_loader = DataLoader(train_graphs, batch_size=128, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=128, shuffle=False)

print(f"Total graphs: {len(train_graphs) + len(test_graphs)} | Train: {len(train_graphs)} | Test: {len(test_graphs)}")


edge_index shape: torch.Size([2, 133])
edge_attr shape: torch.Size([133, 1])
Total graphs: 1251 | Train: 1000 | Test: 251


In [ ]:
class GraphClassifier(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_outputs,
        num_nodes,
        heads,
        num_gat_layers,
        use_gat,
        use_node_emb,
        node_emb_dim,
        concat_raw,
        mlp_hidden,
        dropout,
    ):
        super().__init__()
        self.num_nodes = num_nodes
        self.use_gat = use_gat
        self.num_gat_layers = num_gat_layers
        self.concat_raw = concat_raw
        self.dropout = dropout

        if use_node_emb:
            self.node_emb = Embedding(num_nodes, node_emb_dim)
        else:
            self.node_emb = None
            node_emb_dim = 0

        in_dim = in_channels + node_emb_dim

        if use_gat:
            self.conv1 = GATv2Conv(
                in_dim,
                hidden_channels,
                heads=heads,
                concat=False,
                dropout=dropout,
                edge_dim=1,
                add_self_loops=False,
            )
            if num_gat_layers == 2:
                self.conv2 = GATv2Conv(
                    hidden_channels,
                    hidden_channels,
                    heads=1,
                    concat=False,
                    dropout=dropout,
                    edge_dim=1,
                    add_self_loops=False,
                )
            else:
                self.conv2 = None
        else:
            self.node_proj = Linear(in_dim, hidden_channels)
            self.conv1 = None
            self.conv2 = None

        feat_dim = hidden_channels + (in_channels if concat_raw else 0)
        mlp_layers = []
        prev_dim = feat_dim * num_nodes
        for h in mlp_hidden:
            mlp_layers.append(Linear(prev_dim, h))
            mlp_layers.append(nn.LeakyReLU(negative_slope=0.01))
            prev_dim = h
        mlp_layers.append(Linear(prev_dim, num_outputs))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, x, edge_index, edge_attr, batch):
        x_raw = x
        batch_size = int(batch.max().item()) + 1

        if self.node_emb is not None:
            node_ids = torch.arange(self.num_nodes, device=x.device).repeat(batch_size)
            node_emb = self.node_emb(node_ids)
            x_in = torch.cat([x, node_emb], dim=1)
        else:
            x_in = x

        if self.use_gat:
            x = self.conv1(x_in, edge_index, edge_attr)
            x = F.leaky_relu(x, negative_slope=0.01)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if self.conv2 is not None:
                x = self.conv2(x, edge_index, edge_attr)
                x = F.leaky_relu(x, negative_slope=0.01)
        else:
            x = self.node_proj(x_in)
            x = F.leaky_relu(x, negative_slope=0.01)
            x = F.dropout(x, p=self.dropout, training=self.training)

        if self.concat_raw:
            x = torch.cat([x, x_raw], dim=1)

        x = x.reshape(batch_size, self.num_nodes, -1)
        x = x.reshape(batch_size, -1)
        return self.mlp(x)


def build_model():
    variant = MODEL_VARIANT.lower()
    if variant == "gat1":
        use_gat = True
        num_gat_layers = 1
    elif variant == "gat2":
        use_gat = True
        num_gat_layers = 2
    elif variant == "nogat":
        use_gat = False
        num_gat_layers = 0
    else:
        raise ValueError(f"Unknown MODEL_VARIANT: {MODEL_VARIANT}")

    return GraphClassifier(
        in_channels=1,
        hidden_channels=HIDDEN_CHANNELS,
        num_outputs=num_targets,
        num_nodes=num_nodes,
        heads=HEADS,
        num_gat_layers=num_gat_layers,
        use_gat=use_gat,
        use_node_emb=USE_NODE_EMB,
        node_emb_dim=NODE_EMB_DIM,
        concat_raw=CONCAT_RAW,
        mlp_hidden=MLP_HIDDEN,
        dropout=DROPOUT,
    )


In [5]:
# Train / evaluate
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model().to(device)

print(f"Model variant: {MODEL_VARIANT}")
print(model)

# Higher LR, minimal regularization
optimizer = torch.optim.Adam(model.parameters(), lr=8e-4, weight_decay=0.0)

# Multi-label loss (equivalent to Flux.logitbinarycrossentropy)
def loss_fn(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets, reduction="mean")


def run_epoch(loader, train=False):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for data in loader:
        data = data.to(device)
        if train:
            optimizer.zero_grad()

        logits = model(data.x, data.edge_index, data.edge_attr, data.batch)
        loss = loss_fn(logits, data.y)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * data.y.numel()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        correct += (preds == data.y).sum().item()
        total += data.y.numel()

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


num_epochs = 50
history = []
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    test_loss, test_acc = run_epoch(test_loader, train=False)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_micro_acc": train_acc,
            "test_loss": test_loss,
            "test_micro_acc": test_acc,
        }
    )

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} Micro-Acc: {train_acc:.3f} | "
        f"Test Loss: {test_loss:.4f} Micro-Acc: {test_acc:.3f}"
    )

# history_df = pd.DataFrame(history)
# history_out_path = r"C:\Users\ellio\git\gcn-dmpc\supervised-edge-prediction\ML-analysis\ML-training-history-GCN-floats.csv"
# history_df.to_csv(history_out_path, index=False)
# print(f"Saved training history to: {history_out_path}")


def collect_pred_probs(loader):
    preds = []
    model.eval()
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            logits = model(data.x, data.edge_index, data.edge_attr, data.batch)
            probs = torch.sigmoid(logits)
            preds.append(probs.cpu().numpy())
    return np.vstack(preds)


train_pred_mat = collect_pred_probs(train_loader)
test_pred_mat = collect_pred_probs(test_loader)

# Build output DataFrames: first 21 feature columns, last 210 probability columns
train_pred_features_df = pd.DataFrame(train_features, columns=feature_cols)
train_pred_targets_df = pd.DataFrame(train_pred_mat, columns=target_cols)
train_output_df = pd.concat([train_pred_features_df, train_pred_targets_df], axis=1)

train_out_path = r"C:\Users\ellio\git\gcn-dmpc\final-analysis\first-6-decompositions\MLE-classifier-on-1-6\ML-preds-GCN-train-floats.csv"
train_output_df.to_csv(train_out_path, index=False)
print(f"Saved train predictions to: {train_out_path}")

pred_features_df = pd.DataFrame(test_features, columns=feature_cols)
pred_targets_df = pd.DataFrame(test_pred_mat, columns=target_cols)
output_df = pd.concat([pred_features_df, pred_targets_df], axis=1)

test_out_path = r"C:\Users\ellio\git\gcn-dmpc\final-analysis\first-6-decompositions\MLE-classifier-on-1-6\ML-preds-GCN-floats.csv"
output_df.to_csv(test_out_path, index=False)
print(f"Saved test predictions to: {test_out_path}")


Model variant: gat1
GraphClassifier(
  (node_emb): Embedding(21, 8)
  (conv1): GATv2Conv(9, 256, heads=4)
  (mlp): Sequential(
    (0): Linear(in_features=5397, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=256, out_features=210, bias=True)
  )
)
Epoch 001 | Train Loss: 0.5810 Micro-Acc: 0.752 | Test Loss: 0.3659 Micro-Acc: 0.851
Epoch 002 | Train Loss: 0.3190 Micro-Acc: 0.860 | Test Loss: 0.3027 Micro-Acc: 0.834
Epoch 003 | Train Loss: 0.2837 Micro-Acc: 0.847 | Test Loss: 0.2845 Micro-Acc: 0.857
Epoch 004 | Train Loss: 0.2748 Micro-Acc: 0.865 | Test Loss: 0.2774 Micro-Acc: 0.859
Epoch 005 | Train Loss: 0.2706 Micro-Acc: 0.866 | Test Loss: 0.2760 Micro-Acc: 0.859
Epoch 006 | Train Loss: 0.2691 Micro-Acc: 0.866 | Test Loss: 0.2750 Micro-Acc: 0.859
Epoch 007 | Train Loss: 0.2677 Micro-Acc: 0.866 | Test Loss: 0.2740 Micro-Acc: 0.859
Epoch